# ASL Image Models: HOG + LinearSVM and MobileNetV2 CNN

Trains **Model A** (HOG + LinearSVM) and **Model C** (MobileNetV2) for the three-paradigm comparison.

---

## How to run (Colab browser)

### Before you start
You need a **Kaggle API token** (`kaggle.json`).
- kaggle.com → profile picture → **Settings** → **API** → **Create New Token**
- This downloads `kaggle.json` to your computer.

### Steps
1. Go to [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** → select this file.
2. In the left sidebar, click the **folder icon (📁)** → click the **upload arrow (⬆)** → upload `kaggle.json`. Wait until it appears as `/content/kaggle.json`.
3. Click **Runtime → Run all** (Ctrl+F9). Everything runs automatically.
4. At the end, a download cell triggers browser downloads for all output files.

> **Disconnected mid-run?** Click Runtime → Run all again. HOG features are cached so extraction won't repeat.

In [ ]:
# ── CELL 1: RUNTIME SETUP (run once per session) ─────────────────────────
# Authenticates Kaggle, downloads the image dataset, clones the repo.
# Nothing to edit — just run it.
import json, os, subprocess
from pathlib import Path

# 1a. Find kaggle.json
creds = None
for candidate in [
    Path("/content/kaggle.json"),
    Path.home() / ".kaggle" / "kaggle.json",
    Path("kaggle.json"),
]:
    if candidate.exists():
        creds = json.loads(candidate.read_text())
        print(f"Found kaggle.json at {candidate}")
        break

if creds is None:
    print("kaggle.json not found — a file-picker will appear. Upload your kaggle.json.")
    from google.colab import files as _f
    uploaded = _f.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("Wrong file uploaded. Please upload kaggle.json.")
    creds = json.loads(uploaded["kaggle.json"])

# 1b. Write credentials
os.environ["KAGGLE_USERNAME"] = creds["username"]
os.environ["KAGGLE_KEY"]      = creds["key"]
kdir = Path.home() / ".kaggle"
kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps(creds))
(kdir / "kaggle.json").chmod(0o600)
print(f"Authenticated as: {creds['username']}")

# 1c. Download dataset
subprocess.run(["pip", "install", "kaggle", "-q"], check=True)
if not Path("/content/asl_alphabet_train").exists():
    print("Downloading ASL Alphabet dataset (~1 GB) ...")
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", "grassknoted/asl-alphabet",
         "-p", "/content", "--unzip"], check=True)
    print("Dataset ready.")
else:
    print("Dataset already present.")

# 1d. Clone repo
if not Path("/content/Sign-Language-Project").exists():
    print("Cloning repo ...")
    subprocess.run(["git", "clone",
        "https://github.com/SamrawitDawit/Sign-Language-Project",
        "/content/Sign-Language-Project"], check=True)
else:
    subprocess.run(["git", "-C", "/content/Sign-Language-Project", "pull"], check=True)

data_dir = Path("/content/Sign-Language-Project/data")
for f in ["splits.npz", "landmarks.npy", "landmark_metadata.csv", "label_to_index.json"]:
    print(f"  {'OK    ' if (data_dir/f).exists() else 'MISSING'}  {f}")
print("\nSetup complete.")

In [ ]:
# ── CELL 2: INSTALL PACKAGES ─────────────────────────────────────────────
import subprocess, sys
for pkg in ["scikit-image", "tqdm", "seaborn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Packages ready.")

In [ ]:
# ── CELL 3: IMPORTS ──────────────────────────────────────────────────────
import json, time
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from skimage.feature import hog
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Imports OK")

---
## Part 1 — Configuration

No changes needed if you ran the setup cell above.

In [ ]:
# ── CELL 4: CONFIGURATION ────────────────────────────────────────────────
DATASET_ROOT = "/content"                                     # images at /content/asl_alphabet_train/
DATA_DIR     = Path("/content/Sign-Language-Project/data")   # from cloned repo
OUT_DIR      = Path("/content/outputs")
OUT_DIR.mkdir(exist_ok=True)

# HOG parameters
HOG_SIZE            = 64
HOG_ORIENTATIONS    = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_BLOCK_NORM      = "L2-Hys"

print(f"DATA_DIR exists : {DATA_DIR.exists()}")
print(f"OUT_DIR  exists : {OUT_DIR.exists()}")
print(f"Images   exist  : {Path(DATASET_ROOT, 'asl_alphabet_train').exists()}")

---
## Part 2 — Load Data and Build Split

In [ ]:
# ── CELL 5: LOAD METADATA AND LABEL MAP ──────────────────────────────────
meta = pd.read_csv(DATA_DIR / "landmark_metadata.csv")

with open(DATA_DIR / "label_to_index.json") as f:
    label_to_index: dict = json.load(f)
index_to_label = {v: k for k, v in label_to_index.items()}
num_classes    = len(label_to_index)
class_names    = [index_to_label[i] for i in range(num_classes)]

print(f"Metadata rows : {len(meta)}")
print(f"Classes       : {num_classes}  →  {class_names}")

In [ ]:
# ── CELL 6: BUILD STRATIFIED SPLIT ───────────────────────────────────────
# Uses the same 70/15/15 proportions as the landmark models.
# Rows with labels not in label_to_index (e.g. the 'nothing' class from the
# raw Kaggle dataset) are filtered out first.

all_y_mapped = meta["label"].map(label_to_index)
valid        = all_y_mapped.notna()

dropped = meta["label"][~valid].unique().tolist()
if dropped:
    print(f"Filtering {(~valid).sum()} rows with unknown labels: {dropped}")

all_y = all_y_mapped[valid].values.astype(np.int64)
orig  = np.where(valid)[0]       # maps back to original metadata row numbers
all_i = np.arange(len(orig))

tv_i, te_i = train_test_split(all_i, test_size=0.15,      stratify=all_y,       random_state=42)
tr_i, vl_i = train_test_split(tv_i,  test_size=0.15/0.85, stratify=all_y[tv_i], random_state=42)

tr_idx = orig[tr_i].tolist()
vl_idx = orig[vl_i].tolist()
te_idx = orig[te_i].tolist()
y_tr   = all_y[tr_i]
y_vl   = all_y[vl_i]
y_te   = all_y[te_i]

print(f"Split — train:{len(tr_idx)}  val:{len(vl_idx)}  test:{len(te_idx)}")

In [ ]:
# ── CELL 7: IMAGE PATH REMAPPER + VERIFY ────────────────────────────────
# CSV has paths like /kaggle/input/<dataset>/asl_alphabet_train/...
# We strip the prefix and prepend DATASET_ROOT.

def img_path(meta_row_idx: int) -> Path:
    raw   = meta.iloc[meta_row_idx]["image_path"]
    parts = Path(raw).parts
    try:
        ki  = parts.index("input")
        rel = Path(*parts[ki + 2:])   # skip 'input' and dataset folder name
    except ValueError:
        rel = Path(*parts[1:])        # strip leading '/'
    return Path(DATASET_ROOT) / rel

print("Verifying 5 sample image paths:")
ok = 0
for i in tr_idx[:5]:
    p = img_path(i)
    exists = p.exists()
    ok += exists
    print(f"  {'OK     ' if exists else 'MISSING'}  {p}")

if ok == 0:
    print("\n*** No images found. Did the dataset download in the setup cell? ***")

---
## Part 3 — HOG + LinearSVM

### 3A — Feature Extraction

Each image: grayscale → resize 64×64 → HOG descriptor.  
Features are cached to `hog_features.npz` — re-runs load the cache instead of re-extracting.

In [ ]:
# ── CELL 8: HOG FUNCTION + TIME ESTIMATE ─────────────────────────────────
def extract_hog(path: Path) -> np.ndarray | None:
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (HOG_SIZE, HOG_SIZE))
    return hog(img,
               orientations=HOG_ORIENTATIONS,
               pixels_per_cell=HOG_PIXELS_PER_CELL,
               cells_per_block=HOG_CELLS_PER_BLOCK,
               block_norm=HOG_BLOCK_NORM)

t0 = time.time()
sample_feat = None
for i in tr_idx[:100]:
    f = extract_hog(img_path(i))
    if f is not None and sample_feat is None:
        sample_feat = f
ms_each   = (time.time() - t0) / 100 * 1000
total_min = ms_each * (len(tr_idx) + len(vl_idx) + len(te_idx)) / 60_000
print(f"HOG dim    : {len(sample_feat) if sample_feat is not None else 'N/A'}")
print(f"Speed      : {ms_each:.1f} ms/image")
print(f"Est. total : {total_min:.1f} min for all {len(tr_idx)+len(vl_idx)+len(te_idx)} images")

In [ ]:
# ── CELL 9: EXTRACT HOG FEATURES (WITH CACHE) ────────────────────────────
HOG_CACHE = OUT_DIR / "hog_features.npz"

if HOG_CACHE.exists():
    print("Loading cached HOG features ...")
    c = np.load(HOG_CACHE)
    H_tr, H_vl, H_te = c["H_tr"], c["H_vl"], c["H_te"]
    y_tr, y_vl, y_te = c["y_tr"], c["y_vl"], c["y_te"]
else:
    def extract_split(idxs, labels, desc):
        X, Y, skipped = [], [], 0
        for i, y in tqdm(zip(idxs, labels), total=len(idxs), desc=desc):
            f = extract_hog(img_path(i))
            if f is not None:
                X.append(f); Y.append(y)
            else:
                skipped += 1
        if skipped:
            print(f"  Skipped {skipped} unreadable images")
        return np.array(X, dtype=np.float32), np.array(Y, dtype=np.int64)

    H_tr, y_tr = extract_split(tr_idx, y_tr, "train")
    H_vl, y_vl = extract_split(vl_idx, y_vl, "val  ")
    H_te, y_te = extract_split(te_idx, y_te, "test ")
    np.savez_compressed(HOG_CACHE,
                        H_tr=H_tr, H_vl=H_vl, H_te=H_te,
                        y_tr=y_tr, y_vl=y_vl, y_te=y_te)
    print(f"Saved cache → {HOG_CACHE}")

print(f"train:{H_tr.shape}  val:{H_vl.shape}  test:{H_te.shape}")

### 3B — Training

In [ ]:
# ── CELL 10: TRAIN HOG + LINEARSVC ───────────────────────────────────────
print("Fitting HOG + LinearSVC pipeline ...")
t0 = time.time()
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=5000), cv=3)),
])
pipeline.fit(H_tr, y_tr)
elapsed_hog = time.time() - t0
val_acc_hog = accuracy_score(y_vl, pipeline.predict(H_vl))
print(f"Done in {elapsed_hog:.1f} s")
print(f"Val accuracy : {val_acc_hog:.4f}")

### 3C — Evaluation

In [ ]:
# ── CELL 11: EVALUATE HOG + SVM ──────────────────────────────────────────
hog_preds    = pipeline.predict(H_te)
test_acc_hog = accuracy_score(y_te, hog_preds)
print(f"Test accuracy : {test_acc_hog:.4f}\n")
print(classification_report(y_te, hog_preds, target_names=class_names))

In [ ]:
# ── CELL 12: CONFUSION MATRIX ────────────────────────────────────────────
cm = confusion_matrix(y_te, hog_preds)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"HOG + LinearSVM — Test Accuracy {test_acc_hog:.2%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_hog_svm.png", dpi=120)
plt.show()

In [ ]:
# ── CELL 13: SAVE HOG MODEL + RESULTS ────────────────────────────────────
joblib.dump(pipeline, OUT_DIR / "model_hog_svm.joblib")
results_hog = {
    "hog_svm": {
        "test_accuracy":  round(float(test_acc_hog), 4),
        "val_accuracy":   round(float(val_acc_hog), 4),
        "train_time_s":   round(elapsed_hog, 1),
        "hog_dim":        int(H_tr.shape[1]),
        "hog_image_size": HOG_SIZE,
    }
}
with open(OUT_DIR / "results_hog_svm.json", "w") as f:
    json.dump(results_hog, f, indent=2)
print("Saved model_hog_svm.joblib + results_hog_svm.json")
print(json.dumps(results_hog, indent=2))

---
## Part 4 — MobileNetV2 CNN

Transfer learning from ImageNet-pretrained MobileNetV2. Full fine-tuning, lr=1e-4, 10 epochs.

### 4A — Dataset and DataLoaders

In [ ]:
# ── CELL 14: CNN DATASET ─────────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class ASLDataset(Dataset):
    def __init__(self, indices, labels, transform):
        self.indices   = indices
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img = Image.open(img_path(self.indices[i])).convert("RGB")
        return self.transform(img), int(self.labels[i])


BATCH    = 32
train_ds = ASLDataset(tr_idx, y_tr, train_tf)
val_ds   = ASLDataset(vl_idx, y_vl, eval_tf)
test_ds  = ASLDataset(te_idx, y_te, eval_tf)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"Train  : {len(train_ds)}  Val : {len(val_ds)}  Test : {len(test_ds)}")

### 4B — Model and Training

In [ ]:
# ── CELL 15: TRAIN MOBILENETV2 ───────────────────────────────────────────
model = models.mobilenet_v2(weights="DEFAULT")
model.classifier[1] = nn.Linear(1280, num_classes)
model = model.to(device)
print(f"MobileNetV2: {sum(p.numel() for p in model.parameters()):,} parameters")

optimizer    = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn      = nn.CrossEntropyLoss()
CNN_EPOCHS   = 10
best_val_acc = 0.0
t0           = time.time()

for epoch in range(1, CNN_EPOCHS + 1):
    model.train()
    for xb, yb in tqdm(train_dl, desc=f"Epoch {epoch}/{CNN_EPOCHS}", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    v_acc = accuracy_score(trues, preds)
    print(f"  epoch {epoch:2d} | val_acc {v_acc:.4f}")
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({"state_dict": model.state_dict(), "num_classes": num_classes},
                   OUT_DIR / "model_cnn.pt")

elapsed_cnn = time.time() - t0
print(f"\nDone in {elapsed_cnn:.1f} s  |  Best val acc: {best_val_acc:.4f}")

### 4C — Evaluation

In [ ]:
# ── CELL 16: EVALUATE CNN ────────────────────────────────────────────────
ckpt = torch.load(OUT_DIR / "model_cnn.pt", map_location=device, weights_only=True)
model.load_state_dict(ckpt["state_dict"])
model.eval()

preds, trues = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_dl, desc="Test eval"):
        preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        trues.extend(yb.numpy())

test_acc_cnn = accuracy_score(trues, preds)
print(f"CNN Test accuracy: {test_acc_cnn:.4f}\n")
print(classification_report(trues, preds, target_names=class_names))

In [ ]:
# ── CELL 17: CNN CONFUSION MATRIX ────────────────────────────────────────
cm_cnn = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cm_cnn, annot=False, cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"MobileNetV2 CNN — Test Accuracy {test_acc_cnn:.2%}")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_cnn.png", dpi=120)
plt.show()

In [ ]:
# ── CELL 18: SAVE CNN RESULTS ────────────────────────────────────────────
results_cnn = {
    "cnn": {
        "test_accuracy":     round(float(test_acc_cnn), 4),
        "best_val_accuracy": round(float(best_val_acc), 4),
        "train_time_s":      round(elapsed_cnn, 1),
        "epochs":            CNN_EPOCHS,
        "architecture":      "MobileNetV2 (ImageNet pretrained, full fine-tune)",
        "input_size":        224,
    }
}
with open(OUT_DIR / "results_cnn.json", "w") as f:
    json.dump(results_cnn, f, indent=2)
print("Saved model_cnn.pt + results_cnn.json")
print(json.dumps(results_cnn, indent=2))

---
## Part 5 — Download Outputs

Running the cell below triggers browser downloads for all output files.

**Copy downloaded files into the repo:**

| File | Destination |
|---|---|
| `results_hog_svm.json` | repo root |
| `results_cnn.json` | repo root |
| `model_hog_svm.joblib` | `models/` |
| `model_cnn.pt` | `models/` |
| `confusion_matrix_hog_svm.png` | `docs/` |
| `confusion_matrix_cnn.png` | `docs/` |

In [ ]:
# ── CELL 19: DOWNLOAD OUTPUTS ────────────────────────────────────────────
from google.colab import files as _colab_files

for fname in [
    "results_hog_svm.json", "results_cnn.json",
    "model_hog_svm.joblib", "model_cnn.pt",
    "confusion_matrix_hog_svm.png", "confusion_matrix_cnn.png",
]:
    path = OUT_DIR / fname
    if path.exists():
        _colab_files.download(str(path))
        print(f"Downloading {fname}")
    else:
        print(f"Skipping    {fname}  (not found — did the training cell complete?)")